# LangChain Workflow

#### _Prompting an LLM can develop AI applications much faster._

LangChain:
- Open source framework to develop LLM applications
- 2 packages - python and javascript
- Focuses on composition and modularity

Components of LangChain:
1. Models - language models
2. Prompts - style of creating inputs to pass into the models
3. Indexes - injest data to combine with models
4. Chains - end to end use cases
5. Agents - end time use case which uses the model as a reasoning engine

#### <b>Models, Prompts and Parsers:</b>
- Models refers to the language models
- Prompts are the style of creating inputs to pass into the models
- Parsers refers to the outputs and parses it into a more structured format


<b>LangChain Prompt Templates:</b>
- Using prompt template instead of f string and substitute values directly while calling LLM chat
- It is a useful abstraction to help reuse good prompts when building complex applications that has long and detailed prompts
- Resuse prompts
- LangChain provides prompts for common operations

<b>Output Parsing:</b>
- Aspect of LangChain prompt libraries is output parsing
- While Building complex app, the LLM output is preferably generated in a certain format such as using keywords
- Can extract LLM thought process using keywords for Chain-of-Thought reasoning (ReAct)
- Can ask LLM to output JSON and use LangChain to parse the output

#### _Helper Function to Connect to LLM:_

    import os
    os.environ['ACCESS_TOKEN_NAME'] = 'insert_access_token'

    from openai import OpenAI

    client = OpenAI(
        base_url="https://router.huggingface.co/v1",
        api_key=os.environ["HF_ACCESS_TOKEN"],
    )

    def get_completion(messages, model="zai-org/GLM-5.3:fireworks-ai", temperature=0):
        response = client.chat.completions.create(
            model= model,
            messages = messages,
            temperature = temperature,
        )

        return response.choices[0].message.content

#### <b>_Example Prompt without LangChain:_</b>

Substitue input variable into the prompt using f sting:

    customer_email = """
    Arrr, I be fuming that me blender lid \
    flew off and splattered me kitchen walls \
    with smoothie! And to make matters worse,\
    the warranty don't cover the cost of \
    cleaning up me kitchen. I need yer help \
    right now, matey!
    """

    style = """American English \
    in a calm and respectful tone
    """

    prompt = f"""Translate the text \
    that is delimited by triple backticks 
    into a style that is {style}.
    text: ```{customer_email}```
    """

    print(prompt)

#### <b>_Example Prompt with Chat API - LangChain:_<b>

Model - specify language model to use:

    from langchain.chat_models import ChatOpenAI
    
    chat = ChatOpenAI(temperature=0.0, model=llm_model)
    chat

Prompt Template:
- Use without f string
- Shows input variable used in the prompt and build template that can be resused

    template_string = """Translate the text \
    that is delimited by triple backticks \
    into a style that is {style}. \
    text: ```{text}```
    """

    from langchain.prompts import ChatPromptTemplate

    prompt_template = ChatPromptTemplate.from_template(template_string)

    prompt_template.messages[0].prompt
    prompt_template.messages[0].prompt.input_variables

Format the prompt template with the input variables:

    customer_messages = prompt_template.format_messages(
                        style=customer_style,
                        text=customer_email)

    print(customer_messages[0])

Call the LLM to translate to the style of the customer message:

    customer_response = chat(customer_messages)
    print(customer_response.content)

### <b>_Output Parsers:_</b>

LangChain Parsing:
- LLM is asked to output in json format with specific keys but the type of output generated by the LLM is a string
- Convert str to json to extract key values using LangChain's output parsers to import response scheme and structured output parser
- Format instructions that LLM should return output

##### <b>_Example LangChain Output Parsing:_</b>

    customer_review = """\
    This leaf blower is pretty amazing.  It has four settings:\
    candle blower, gentle breeze, windy city, and tornado. \
    It arrived in two days, just in time for my wife's \
    anniversary present. \
    I think my wife liked it so much she was speechless. \
    So far I've been the only one using it, and I've been \
    using it every other morning to clear the leaves on our lawn. \
    It's slightly more expensive than the other leaf blowers \
    out there, but I think it's worth it for the extra features.
    """

    review_template = """\
    For the following text, extract the following information:

    gift: Was the item purchased as a gift for someone else? \
    Answer True if yes, False if not or unknown.

    delivery_days: How many days did it take for the product \
    to arrive? If this information is not found, output -1.

    price_value: Extract any sentences about the value or price,\
    and output them as a comma separated Python list.

    Format the output as JSON with the following keys:
    gift
    delivery_days
    price_value

    text: {text}
    """

LangChain Prompts:

    from langchain.prompts import ChatPromptTemplate

    prompt_template = ChatPromptTemplate.from_template(review_template)
    print(prompt_template)

    messages = prompt_template.format_messages(text=customer_review)
    chat = ChatOpenAI(temperature=0.0, model=llm_model)
    response = chat(messages)
    print(response.content)

Type of LLM response is String so will not be able to parse key values like a dictionary:

    type(response.content)

##### <b> Parse the LLM output string into a Python dictionary: </b>

    from langchain.output_parsers import ResponseSchema
    from langchain.output_parsers import StructuredOutputParser

    gift_schema = ResponseSchema(name="gift",
                                description="Was the item purchased\
                                as a gift for someone else? \
                                Answer True if yes,\
                                False if not or unknown.")
    delivery_days_schema = ResponseSchema(name="delivery_days",
                                        description="How many days\
                                        did it take for the product\
                                        to arrive? If this \
                                        information is not found,\
                                        output -1.")
    price_value_schema = ResponseSchema(name="price_value",
                                        description="Extract any\
                                        sentences about the value or \
                                        price, and output them as a \
                                        comma separated Python list.")

    response_schemas = [gift_schema, 
                        delivery_days_schema,
                        price_value_schema]

    output_parser = StructuredOutputParser.from_response_schemas(response_schemas)

Format instuctions for LLM to return output in a specific manner:

    format_instructions = output_parser.get_format_instructions()
    print(format_instructions)

Create new prompt that takes in both text and format instructions as input variables:

    review_template_2 = """\
    For the following text, extract the following information:

    gift: Was the item purchased as a gift for someone else? \
    Answer True if yes, False if not or unknown.

    delivery_days: How many days did it take for the product\
    to arrive? If this information is not found, output -1.

    price_value: Extract any sentences about the value or price,\
    and output them as a comma separated Python list.

    text: {text}

    {format_instructions}
    """

Create Prompt template and format it using the input varaibles:

    prompt = ChatPromptTemplate.from_template(template=review_template_2)

    messages = prompt.format_messages(text=customer_review, 
                                    format_instructions=format_instructions)

Call LLM with prompt message created:

    response = chat(messages)

    print(response.content)

Parse output the LLM chat response in json:

    output_dict = output_parser.parse(response.content)

    output_dict.get('delivery_days')